# İKY Kitaplarının İçindekiler Sayfalarıyla *hrdergi.com* Makalelerinin Analizi

**Amaç:** 8 adet İnsan Kaynakları Yönetimi ders kitabının *İçindekiler* sayfalarından alan "kavram haritası" çıkarmak ve bu haritayı `hrdergi_korpus_final.csv` içindeki **Makale İçeriği** sütunundaki ~3.900 makaleye uygulamak.

**Akış**
1. Veri yükleme ve temizlik
2. Kitap txt dosyalarından içindekiler (İÇ) bloğunu çıkarma ve başlıkları ayrıştırma
3. İÇ'ten türetilen (a) konu sözlüğü ve (b) konsensüs kavramlar
4. Makale metinlerinin ön işlemesi
5. Konu/kavram yaygınlığı: genel, yıllara/dönemlere göre
6. İÇ başlıkları ↔ makaleler eşleştirmesi (TF-IDF kosinüs): hangi kitap konuları dergide ne kadar işleniyor?
7. Denetimsiz konu modeli (NMF) ve İÇ konularıyla karşılaştırma
8. Dışa aktarım (Excel + grafikler)

> **Not:** Kitapların OCR çıktılarında bozuk karakterler (`Ġ`→`İ`, `ġ`→`Ş`, `Ģ`→`ş`) ve düzensiz biçim vardır. İçindekiler satır aralıkları aşağıdaki `BOOKS` tablosunda **elle düzenlenebilir**. `0` numaralı kitabın (4 KB) içindekiler sayfası boş olduğundan atlanır.

## 0. Kurulum ve yollar

In [ ]:
# Colab'da gerekirse: !pip -q install pandas scikit-learn matplotlib openpyxl
import os, re, glob, zipfile, unicodedata, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90, "display.width", 160)

# ---- YOLLAR: kendi ortamınıza göre değiştirin ----
ARCHIVE_ZIP = "Archive.zip"                      # kitap txt dosyaları
CSV_ZIP     = "hrdergi_korpus_final.csv.zip"     # makale korpusu (veya doğrudan .csv)
WORK_DIR    = "work"                             # açılan dosyalar ve çıktılar
os.makedirs(WORK_DIR, exist_ok=True)

def unzip(path, dest):
    # Zip içindeki dosya adları (macOS) bozuk kodlanmış olabilir: adı onar, kısalt, NFC'ye çevir
    if not os.path.exists(path):
        return
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(path) as z:
        for zi in z.infolist():
            if zi.is_dir() or "__MACOSX" in zi.filename:
                continue
            name = zi.filename
            if not (zi.flag_bits & 0x800):                       # UTF-8 bayrağı yoksa cp437 okunmuştur
                try: name = name.encode("cp437").decode("utf-8")
                except Exception: pass
            name = unicodedata.normalize("NFC", os.path.basename(name))
            stem, ext = os.path.splitext(name)
            while len(stem.encode("utf-8")) > 180: stem = stem[:-1]   # dosya sistemi sınırı için kısalt
            name = stem + ext
            with z.open(zi) as src, open(os.path.join(dest, name), "wb") as dst:
                dst.write(src.read())
unzip(ARCHIVE_ZIP, f"{WORK_DIR}/books")
unzip(CSV_ZIP, f"{WORK_DIR}/csv")

# __MACOSX gibi meta klasörleri dışla; dosya adlarını sıralı al
BOOK_FILES = sorted(p for p in glob.glob(f"{WORK_DIR}/books/**/*.txt", recursive=True) if "__MACOSX" not in p)
CSV_PATH = next(p for p in glob.glob(f"{WORK_DIR}/csv/**/*.csv", recursive=True) if "__MACOSX" not in p) \
           if glob.glob(f"{WORK_DIR}/csv/**/*.csv", recursive=True) else "hrdergi_korpus_final.csv"
print(len(BOOK_FILES), "kitap dosyası;", "CSV:", CSV_PATH)

## 1. Makale korpusunu yükleme ve temizlik

In [ ]:
df = pd.read_csv(CSV_PATH)
TEXT = "Makale İçeriği"
print(df.shape); print(df.columns.tolist())

df[TEXT] = df[TEXT].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
n0 = len(df)
df = df.drop_duplicates(subset=["Makale Başlığı", TEXT]).reset_index(drop=True)
df["Yıl"] = pd.to_numeric(df["Yıl"], errors="coerce").astype("Int64")
df["uzunluk"] = df[TEXT].str.len()
print(f"Yinelenen kayıt: {n0-len(df)} | Makale sayısı: {len(df)}")
display(df["uzunluk"].describe().round(0))
display(df.groupby("Yıl").size().rename("makale").to_frame().T)

## 2. Kitaplardan İçindekiler bloğunu çıkarma

Her kitap için içindekiler bloğunun `(başlangıç, bitiş)` satırı verilir (0 tabanlı). Bunlar dosyalar incelenerek belirlenmiştir; farklı baskı kullanırsanız burayı güncelleyin.

In [ ]:
# dosya sırası = sorted(BOOK_FILES) sırası. (None, None) -> içindekiler yok/boş, atla.
def tr_lower(s):
    return s.replace("İ", "i").replace("I", "ı").lower()

BOOKS = {
    0: dict(ad="İKY (Bingöl, 8. bs.)",             start=None, end=None),   # yalnızca 4 KB, içindekiler boş
    1: dict(ad="İKY (Tüzüner vd., 2023, Beta)",    start=262,  end=1761),
    2: dict(ad="İKY (Çetin-Dinç Elmalı-Arslan, 2019)", start=207, end=1227),
    3: dict(ad="İKY (Bakan, 2014, Gazi)",          start=235,  end=1042),
    4: dict(ad="İKY (Saruhan-Yıldız, 2019, Beta)", start=75,   end=222),
    5: dict(ad="İKY Yoluyla... (Keçecioğlu, 2012)",start=63,   end=100),
    6: dict(ad="Stratejik İKY (Bayraktaroğlu-Atay, 2016)", start=142, end=444),
    7: dict(ad="İKY ve Kariyer (Şimşek vd., 2016)",start=259,  end=470),
}
assert len(BOOK_FILES) >= len(BOOKS), "Kitap dosyası sayısı beklenenden az"
# Dosya adından kitap eşleme (sıra bağımlılığını kaldırır). Anahtar: ad içinde geçen ayırt edici sözcük
KEYS = {0: "Bilinmeyen", 1: "TÜZÜNER", 2: "ÇETİN", 3: "Bakan", 4: "SARUHAN", 5: "Keçecioğlu", 6: "BAYRAKTAROGLU", 7: "Şimşek"}
def find_book(key):
    k = tr_lower(unicodedata.normalize("NFC", key))
    hits = [p for p in BOOK_FILES if k in tr_lower(unicodedata.normalize("NFC", os.path.basename(p)))]
    assert hits, f"Dosya bulunamadı: {key}"
    return hits[0]

def read_lines(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        return f.read().split("\n")

# OCR/yazı tipi kaynaklı bozuk karakter düzeltmesi
FIX = str.maketrans({"Ġ": "İ", "ġ": "Ş", "Ģ": "ş"})
def fix_chars(s): return unicodedata.normalize("NFC", s.translate(FIX))

toc_raw = []
for i, meta in BOOKS.items():
    if meta["start"] is None:
        print(f"[atlandı] {i}: {meta['ad']}"); continue
    L = read_lines(find_book(KEYS[i]))[meta["start"]:meta["end"]]
    toc_raw += [(i, meta["ad"], fix_chars(l)) for l in L if l.strip()]
toc_raw = pd.DataFrame(toc_raw, columns=["kitap_id", "kitap", "satir"])
print(toc_raw.groupby("kitap").size())

### 2.1 Satırları başlıklara ayrıştırma

In [ ]:
def tr_lower(s):
    return s.replace("İ", "i").replace("I", "ı").lower()

LEADER = re.compile(r"[\.…·_\-–]{3,}.*$")          # ..... 15  kuyruğu
TRAIL_PG = re.compile(r"\s+[\divxlcIVXLC]{1,4}(\s*[,-]\s*\d+)*\s*$")  # sondaki sayfa no
LEAD_NUM = re.compile(r"^\s*((\d+\.)+\d*\.?|\(?\d+\)|\(?[A-Za-zÇĞİÖŞÜçğıöşü]{1,3}\s?\)|[A-ZÇĞİÖŞÜ]\.|[IVX]+\.)\s*")   # 1.2. / a) / bf ) / A. / IV.   # 1.2.3. / A. / IV.
CHAP = re.compile(r"^\s*((\d+|[İI]LK|BİRİNCİ|İKİNCİ|ÜÇÜNCÜ|DÖRDÜNCÜ|BEŞİNCİ|ALTINCI|YEDİNCİ|SEKİZİNCİ|DOKUZUNCU|ONUNCU|ON\s?\w+)\.?\s*B[ÖO]L[ÜU]M|B[ÖO]L[ÜU]M\s*\d+|\d+\.\s*B[öo]l[üu]m)\s*$", re.I)
NOISE = re.compile(r"^(şekil|tablo|vaka|okuma parças|kaynakça|kaynaklar|dizin|konu dizini|içindekiler|sayfa no|önsöz|sunuş|teşekkür|giriş$|sonuç$|özet$|kavramlar$|tartışma soruları|test soruları|yazar hakkında|ek \d|prof\.|doç\.|yrd\.|dr\.|arş\.|öğr\.)", re.I)
HDR_ONLY = re.compile(r"^(bölüm|chapter)?\s*\d*$", re.I)

rows, cur_chap = [], defaultdict(lambda: 0)
for r in toc_raw.itertuples(index=False):
    t = r.satir.strip()
    if CHAP.match(t):
        cur_chap[r.kitap_id] += 1; continue
    t = LEADER.sub("", t).strip()
    t = TRAIL_PG.sub("", t).strip()
    t = LEAD_NUM.sub("", t).strip(" .:;,-–")
    if len(t) < 6 or HDR_ONLY.match(t) or NOISE.match(tr_lower(t)): continue
    if len(t.split()) > 25: continue                      # gövde metni sızıntısı
    if re.fullmatch(r"[\W\d_]+", t): continue
    rows.append((r.kitap_id, r.kitap, cur_chap[r.kitap_id], t))

toc = pd.DataFrame(rows, columns=["kitap_id", "kitap", "bolum", "baslik"]).drop_duplicates(["kitap_id", "baslik"]).reset_index(drop=True)
toc["baslik_norm"] = toc["baslik"].map(tr_lower)
print(f"{len(toc)} benzersiz başlık")
display(toc.groupby("kitap").agg(baslik=("baslik", "size"), bolum=("bolum", "max")))
display(toc.sample(15, random_state=1)[["kitap", "bolum", "baslik"]])

## 3. İçindekilerden konu sözlüğü

**(a) Konu sözlüğü:** Kitap bölüm/başlık dizilerinde yinelenen İKY işlevleri (işe alma, performans, ücret...) tanımlanmıştır. Her konu için anahtar ifadeler verilir; ardından **kitap başlıklarında gerçekten geçip geçmediği** sınanır (İÇ doğrulaması). Sözlüğü kendi araştırma sorunuza göre düzenleyebilirsiniz.

**(b) Konsensüs kavramlar:** Başlıklardan 1–3'lü sözcük öbekleri çıkarılır; en az 2 farklı kitapta geçenler "çekirdek müfredat" kavramı sayılır.

In [ ]:
THEMES = {
 "İş analizi, tasarımı ve tanımı":      ["iş analiz", "iş tanım", "iş tasarım", "iş değerleme", "iş zenginleştir", "iş genişlet", "iş rotasyon", "iş gerekleri", "iş etüd"],
 "İK planlaması":                        ["insan kaynakları planla", "işgücü planla", "iş gücü planla", "norm kadro", "talep tahmin"],
 "İşe alma ve seçme":                    ["işe alım", "işe alma", "personel temini", "personel seçimi", "seçme", "mülakat", "mülakât", "görüşme tekni", "aday", "özgeçmiş", "cv ", "işveren markas", "yerleştirme", "oryantasyon", "işe başlatma", "uyum eğitimi"],
 "Eğitim ve geliştirme":                 ["eğitim ihtiyac", "eğitim ve geliştir", "personel eğitimi", "çalışan eğitimi", "yönetici geliştir", "hizmet içi", "mesleki eğitim", "öğrenen örgüt", "e-öğrenme", "koçluk", "mentor", "mentörlük", "işbaşı eğitim"],
 "Performans yönetimi":                  ["performans", "değerlendirme sistem", "360 derece", "hedeflerle yönetim", "dengeli skor", "kpi", "okr", "geri bildirim"],
 "Ücret ve ödül yönetimi":               ["ücret", "maaş", "ödül", "prim", "yan ödeme", "yan hak", "teşvik", "tazminat", "zam ", "bordro", "performansa dayalı ödeme", "esnek ödeme"],
 "Kariyer yönetimi":                     ["kariyer", "terfi", "yükselme", "yedekleme", "halef", "kariyer planla"],
 "Yetenek ve yetkinlik yönetimi":        ["yetenek yönetim", "yetenek savaş", "yetkinlik", "yetenek havuz", "yüksek potansiyel", "talent"],
 "İş sağlığı ve güvenliği":              ["iş sağlığı", "iş güvenliği", "isg", "iş kazası", "meslek hastalığı", "ergonomi", "mobbing", "psikolojik taciz", "stres", "tükenmişlik"],
 "Endüstri ilişkileri ve sendika":       ["sendika", "toplu iş sözleşmesi", "toplu pazarlık", "grev", "lokavt", "endüstri ilişkileri", "çalışma ilişkileri", "iş hukuku", "iş kanunu", "4857", "kıdem tazminat", "işten çıkarma", "işten çıkar", "iş güvencesi", "arabulucu"],
 "Motivasyon, bağlılık ve iş tatmini":   ["motivasyon", "iş tatmini", "iş doyumu", "örgütsel bağlılık", "duygusal bağlılık", "çalışan bağlılığı", "işe bağlılık", "iş yaşam kalitesi", "iş-yaşam dengesi", "çalışan memnuniyet", "çalışan deneyimi", "çalışan katılım", "ödüllendirme"],
 "Örgüt kültürü ve iklimi":              ["örgüt kültür", "örgütsel kültür", "kurum kültür", "örgüt iklimi", "örgütsel iklim", "değerler", "etik"],
 "Liderlik ve yöneticilik":              ["liderlik", "lider ", "yönetici gelişim", "dönüşümcü", "etkileşimci"],
 "Stratejik İKY":                        ["stratejik insan kaynak", "stratejik iky", "stratejik yönetim", "stratejik uyum", "rekabet üstünlüğü", "rekabet avantaj", "kaynak temelli", "kaynak tabanlı", "insan sermayesi", "entelektüel sermaye", "ik skor"],
 "Uluslararası ve küresel İKY":          ["uluslararası insan kaynak", "küresel insan kaynak", "ekspatriye", "expatriate", "uluslararası", "küreselleş", "çok uluslu", "kültürlerarası"],
 "Çeşitlilik, kapsayıcılık, cinsiyet":   ["çeşitlilik", "farklılıkların yönetimi", "kadın", "cinsiyet", "cam tavan", "kapsayıcılık", "engelli", "kuşak", "z kuşağı", "y kuşağı", "toplumsal cinsiyet", "eşitlik"],
 "İK bilgi sistemleri ve dijitalleşme":  ["bilgi sistemi", "ik analitik", "insan kaynakları analitik", "dijital", "yapay zeka", "yapay zekâ", "otomasyon", "e-ik", "veri analiz", "büyük veri", "chatbot", "algoritma", "uzaktan çalışma", "hibrit çalışma", "evden çalışma", "uzaktan çalış"],
 "Esnek çalışma ve istihdam biçimleri":  ["esnek çalışma", "kısmi süreli", "part-time", "taşeron", "alt işveren", "dış kaynak kullan", "outsourcing", "geçici iş", "belirli süreli", "telekomüt", "serbest çalışan", "gig", "kısa çalışma"],
 "Disiplin, çatışma ve işten ayrılma":   ["disiplin", "çatışma", "işten ayrıl", "işten çıkış", "devir hızı", "personel devri", "emeklilik", "işsizlik", "çıkış görüşme", "işgören devri"],
 "Toplumsal sorumluluk ve sürdürülebilirlik": ["sürdürülebilir", "sosyal sorumluluk", "kurumsal sorumluluk", "gönüllü", "esg"],
 "Kriz, pandemi ve değişim yönetimi":    ["pandemi", "covid", "kriz", "değişim yönetim", "kurumsal dönüşüm", "salgın", "deprem", "belirsizlik"],
}
THEME_PATTERNS = {k: [tr_lower(p) for p in v] for k, v in THEMES.items()}

# İÇ doğrulaması: her konunun anahtar ifadeleri kitap başlıklarında kaç kez bulunuyor?
def theme_hits(text):
    return {k: any(p in text for p in pats) for k, pats in THEME_PATTERNS.items()}
toc_theme = pd.DataFrame([theme_hits(t) for t in toc["baslik_norm"]])
toc_theme["kitap"] = toc["kitap"].values
toc_cov = toc_theme.groupby("kitap").sum().T
toc_cov["kitap_sayisi"] = (toc_cov > 0).sum(axis=1)
toc_cov = toc_cov.sort_values("kitap_sayisi", ascending=False)
print("Konu × kitap: başlık sayısı (İÇ doğrulaması)")
display(toc_cov.style.background_gradient(cmap="Blues", axis=None))

In [ ]:
# (b) Konsensüs kavramlar: başlıklardan n-gram, >=2 kitapta geçenler
STOP_TR = set('''ve ile ya da de da bir bu şu o için gibi olarak olan daha çok en ise ama fakat ancak veya hem
her bazı tüm ki mi mı mu mü ne nasıl neden kadar sonra önce üzerine üzerinde arasındaki arasında içinde içerisinde
bölüm bölümü ilişkin ilgili yönelik göre karşı ait ilk son genel temel yeni diğer ayrıca olarak sadece
yönetimi yönetim sistemi kavramı kavram önemi amaçları özellikleri türleri sorunlar sorunları yöntemleri yöntem
süreç süreci rolü rol işlevi işlevleri tanımı çerçevesinde bakış yaklaşım yaklaşımı yaklaşımlar modeller modeli
insan kaynakları kaynağı iky ik'''.split())

GENERIC_PREFIX = ("yönetim", "kaynak", "gelişim", "bileşen", "boyut", "bakış", "uygulama", "değerlendirm")   # çekimli genel sözcükler
def tokens(s):
    return [w for w in re.findall(r"[a-zçğıöşüâîû]{3,}", s) if w not in STOP_TR and not w.startswith(GENERIC_PREFIX[:6])]

def ngrams(toks, nmax=3):
    for n in range(1, nmax + 1):
        for i in range(len(toks) - n + 1):
            yield " ".join(toks[i:i + n])

book_ng = defaultdict(set)
for r in toc.itertuples():
    for g in set(ngrams(tokens(r.baslik_norm))):
        book_ng[g].add(r.kitap_id)
cons = pd.DataFrame([(g, len(b)) for g, b in book_ng.items() if len(b) >= 2], columns=["kavram", "kitap_sayisi"])
cons["n_sozcuk"] = cons["kavram"].str.count(" ") + 1
cons = cons.sort_values(["kitap_sayisi", "n_sozcuk", "kavram"], ascending=[False, False, True]).reset_index(drop=True)
print(len(cons), "konsensüs kavram")
display(cons.head(40))

## 4. Makale metinlerinin ön işlemesi

In [ ]:
STOP_ART = STOP_TR | set('''daha çok olarak olan olduğu olduğunu olmak olması olmasi yani yılında yıl bugün şirket şirketler şirketin
şirketlerin kurum kurumlar çalışan çalışanlar çalışanlara çalışanların iş işte işin işler işletme işletmeler
bunun bunu bunlar bunların onun onu onlar ancak hatta zaten üzere kendi kendini sadece ayrıca ise ise
ilgili gerekir gerekiyor gerekmektedir edilmesi edilir edilen yapılan yapılması yapmak yapılır etmek ediyor
olabilir olabilecek olacak oluyor olur olmuştur bulunmaktadır bulunan artık özellikle ayrı örneğin yüzde
hrdergi türkiye dergi sayı sayısı'''.split())

def prep(s):
    s = tr_lower(s)
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-zçğıöşüâîû0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def crude_stem(w, k=5):
    # Türkçe için basit önek kökleme (F5): ilk k harf
    return w[:k]

df["metin_temiz"] = df[TEXT].map(prep)
df["sozcuk_sayisi"] = df["metin_temiz"].str.split().str.len()
print(df["sozcuk_sayisi"].describe().round(0))
# çok kısa (ör. sadece özet/ilan) makaleleri işaretle
KISA = 80
print("Çok kısa makale (<%d sözcük): %d" % (KISA, (df["sozcuk_sayisi"] < KISA).sum()))
dfa = df[df["sozcuk_sayisi"] >= KISA].reset_index(drop=True)
print("Analize giren makale:", len(dfa))

## 5. Konu yaygınlığı

İki ölçü: **(i) içerme** — makale konuya ait en az bir ifade içeriyor mu; **(ii) yoğunluk** — 1.000 sözcük başına ifade sayısı (uzun makalelerin kayırılmasını engeller).

In [ ]:
def count_patterns(text, pats):
    return sum(text.count(p) for p in pats)

M = pd.DataFrame({k: dfa["metin_temiz"].map(lambda t, p=pats: count_patterns(t, p)) for k, pats in THEME_PATTERNS.items()})
DENS = M.div(dfa["sozcuk_sayisi"], axis=0) * 1000
PRES = (M >= 2)           # en az 2 geçiş: tesadüfi anmayı ele
summary = pd.DataFrame({
    "makale_%": (PRES.mean() * 100).round(1),
    "ort_yoğunluk_/1000": DENS.mean().round(2),
    "toplam_geçiş": M.sum(),
}).sort_values("makale_%", ascending=False)
summary["ic_kitap_sayisi"] = toc_cov["kitap_sayisi"].reindex(summary.index)
display(summary)

fig, ax = plt.subplots(figsize=(8, 7))
summary["makale_%"].sort_values().plot.barh(ax=ax, color="#2f6f9f")
ax.set_xlabel("Konuya en az 2 kez değinen makale oranı (%)"); ax.set_title("hrdergi makalelerinde İKY konuları")
plt.tight_layout(); plt.savefig(f"{WORK_DIR}/konu_yayginligi.png", dpi=150); plt.show()

In [ ]:
# Yıl / dönem kırılımı
dfa["donem"] = pd.cut(dfa["Yıl"].astype(float), bins=[1995, 2004, 2009, 2014, 2019, 2022, 2030],
                      labels=["≤2004", "2005-09", "2010-14", "2015-19", "2020-22", "2023+"])
byp = (PRES.groupby(dfa["donem"]).mean() * 100).T.round(1)
display(byp.style.background_gradient(cmap="YlOrRd", axis=1))

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(byp.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(byp.shape[1])); ax.set_xticklabels(byp.columns)
ax.set_yticks(range(byp.shape[0])); ax.set_yticklabels(byp.index)
for i in range(byp.shape[0]):
    for j in range(byp.shape[1]):
        ax.text(j, i, f"{byp.values[i,j]:.0f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="% makale"); ax.set_title("Konuların dönemlere göre yaygınlığı")
plt.tight_layout(); plt.savefig(f"{WORK_DIR}/konu_donem_isi_haritasi.png", dpi=150); plt.show()

# Yükselen / düşen konular: dönem ilk ↔ son fark
trend = (byp.iloc[:, -1] - byp.iloc[:, 0]).sort_values()
print("En çok düşen:"); display(trend.head(5).round(1))
print("En çok yükselen:"); display(trend.tail(5).round(1))

In [ ]:
# Konsensüs (kitap) kavramlarının makalelerdeki frekansı
top_c = cons[(cons["n_sozcuk"] >= 2) & (cons["kitap_sayisi"] >= 2)].head(80)["kavram"].tolist()   # tek sözcükler fazla genel
big = " || ".join(dfa["metin_temiz"])
cons_freq = pd.DataFrame({"kavram": top_c,
                          "makalede_gecis": [big.count(c) for c in top_c],
                          "makale_sayisi": [int(dfa["metin_temiz"].str.contains(re.escape(c)).sum()) for c in top_c]})
cons_freq = cons_freq.merge(cons[["kavram", "kitap_sayisi"]], on="kavram").sort_values("makale_sayisi", ascending=False)
display(cons_freq.head(30))

## 6. İçindekiler başlıkları ↔ makaleler (TF-IDF kosinüs eşleştirme)

Her kitap başlığı bir "sorgu" sayılır; makalelerle kosinüs benzerliği hesaplanır.
Çıktılar: (i) başlık başına en yakın makale, (ii) bölüm/kitap düzeyinde dergi kapsamı, (iii) hiçbir başlığa benzemeyen makaleler (**kitap-dışı / yeni konular**).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def analyzer(doc):
    toks = [crude_stem(w) for w in doc.split() if len(w) > 2 and w not in STOP_ART]
    return toks + [a + "_" + b for a, b in zip(toks, toks[1:])]

vec = TfidfVectorizer(analyzer=analyzer, min_df=5, max_df=0.5, sublinear_tf=True)
X = vec.fit_transform(dfa["metin_temiz"])
Q = vec.transform(toc["baslik_norm"].map(prep))
print("Makale matrisi:", X.shape, "| Başlık matrisi:", Q.shape)

S = linear_kernel(Q, X)                       # başlık × makale
toc["en_yakin_sim"] = S.max(axis=1)
toc["en_yakin_makale"] = dfa["Makale Başlığı"].values[S.argmax(axis=1)]
THR = np.percentile(S[S > 0], 95)             # veri-güdümlü eşik
toc["eslesen_makale"] = (S >= THR).sum(axis=1)
print(f"Eşik (benzerlik) = {THR:.3f}")
display(toc.sort_values("eslesen_makale", ascending=False).head(20)[["kitap", "baslik", "eslesen_makale", "en_yakin_makale"]])

In [ ]:
cov = toc.groupby(["kitap", "bolum"]).agg(baslik=("baslik", "size"),
                                         ort_sim=("en_yakin_sim", "mean"),
                                         ort_eslesen=("eslesen_makale", "mean")).round(3)
display(cov.sort_values("ort_eslesen", ascending=False).head(15))
print("Dergide en az işlenen kitap başlıkları (ilgili makale çok az):")
display(toc[toc["en_yakin_sim"] > 0].sort_values("eslesen_makale").head(15)[["kitap", "baslik", "en_yakin_sim", "eslesen_makale"]])

# Hiçbir başlıkla eşleşmeyen makaleler
art_best = S.max(axis=0)
dfa["ic_en_yakin_sim"] = art_best
dfa["ic_en_yakin_baslik"] = toc["baslik"].values[S.argmax(axis=0)]
orphan = dfa.sort_values("ic_en_yakin_sim").head(25)[["Yıl", "Makale Başlığı", "ic_en_yakin_sim", "URL"]]
print("Kitap içindekilerine en az benzeyen makaleler (olası yeni/uç konular):")
display(orphan)

## 7. Denetimsiz konu modeli (NMF) ve İÇ konularıyla karşılaştırma

In [ ]:
from sklearn.decomposition import NMF
K = 15
vec2 = TfidfVectorizer(analyzer=lambda d: [w for w in d.split() if len(w) > 3 and w not in STOP_ART],
                       min_df=10, max_df=0.4, sublinear_tf=True, max_features=15000)
X2 = vec2.fit_transform(dfa["metin_temiz"])
nmf = NMF(n_components=K, init="nndsvda", random_state=42, max_iter=400)
W = nmf.fit_transform(X2)
terms = np.array(vec2.get_feature_names_out())
tinfo = pd.DataFrame({
    "konu": range(K),
    "en_belirgin_sozcukler": [", ".join(terms[nmf.components_[k].argsort()[::-1][:10]]) for k in range(K)],
    "makale_payi_%": (np.bincount(W.argmax(axis=1), minlength=K) / len(W) * 100).round(1),
})
display(tinfo)

# NMF konusu ↔ İÇ konusu: ortalama yoğunluk korelasyonu
dfa["nmf_konu"] = W.argmax(axis=1)
cmp_ = DENS.groupby(dfa["nmf_konu"]).mean() / DENS.mean()      # genel ortalamaya oranla (lift)
best = pd.DataFrame({"nmf_konu": cmp_.index,
                     "en_yakin_ic_konusu": cmp_.idxmax(axis=1).values,
                     "lift": cmp_.max(axis=1).round(2).values})
display(best.merge(tinfo[["konu", "en_belirgin_sozcukler"]], left_on="nmf_konu", right_on="konu").drop(columns="konu"))

# NMF konularının yıllara göre seyri
tp = pd.crosstab(dfa["donem"], dfa["nmf_konu"], normalize="index") * 100
ax = tp.plot(kind="area", stacked=True, figsize=(10, 5), colormap="tab20")
ax.set_ylabel("%"); ax.set_title("NMF konularının dönemsel dağılımı"); ax.legend(ncol=3, fontsize=7, bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.savefig(f"{WORK_DIR}/nmf_donem.png", dpi=150); plt.show()

## 8. En sık sözcükler ve dışa aktarım

In [ ]:
# Yıllara göre en ayırt edici sözcükler (dönem başına TF-IDF toplamı)
top = pd.DataFrame({str(d): np.asarray(X2[(dfa["donem"] == d).values].mean(axis=0)).ravel()
                    for d in dfa["donem"].cat.categories}, index=terms)
for d in top.columns:
    print(d, "->", ", ".join(top[d].sort_values(ascending=False).head(12).index))

In [ ]:
out = f"{WORK_DIR}/hrdergi_icindekiler_analiz.xlsx"
with pd.ExcelWriter(out) as xw:
    toc.drop(columns=["baslik_norm"]).to_excel(xw, sheet_name="IC_basliklar", index=False)
    toc_cov.to_excel(xw, sheet_name="IC_konu_dogrulama")
    cons.to_excel(xw, sheet_name="konsensus_kavramlar", index=False)
    summary.to_excel(xw, sheet_name="konu_yayginligi")
    byp.to_excel(xw, sheet_name="konu_donem")
    cons_freq.to_excel(xw, sheet_name="kavram_frekans", index=False)
    cov.to_excel(xw, sheet_name="bolum_kapsam")
    orphan.to_excel(xw, sheet_name="kitap_disi_makaleler", index=False)
    tinfo.to_excel(xw, sheet_name="nmf_konular", index=False)
    dfa.drop(columns=[TEXT, "metin_temiz"]).to_excel(xw, sheet_name="makale_ozet", index=False)
print("Yazıldı:", out)

## Yorum için kontrol listesi
* **Sözlük geçerliği:** `toc_cov` tablosunda hiç/az kitapta geçen konular için anahtar ifadeleri gözden geçirin.
* **Eşik:** `THR` %95 yüzdeliğidir; daha sıkı sonuç için %98 deneyin.
* **Kökleme:** F5 (ilk 5 harf) kaba bir yöntemdir; `zeyrek`/`snowballstemmer` ile değiştirilebilir.
* **Doğrulama:** Konu atamalarını rastgele 30 makalede elle kontrol edip kesinlik oranı raporlayın.
* **Dengesizlik:** Makale sayısı yıllara eşit dağılmaz (bkz. bölüm 1); dönemler arası karşılaştırmada yüzdeleri kullanın.